# System 2: Material Discovery Pipeline

This notebook runs System 2 to find a specific substitute material Z based on material properties W.

**Workflow**: Properties W → Map to Materials → Propose Candidate → Validate → Material Z


## Setup: Import modules and load configuration


In [1]:
import sys
import os
import json
from pathlib import Path

# Add src to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Import modules
from src.config import load_config
from src.utils import llm, TransformerEmbeddingFunction, ChatLogger, MaterialDatabase, SubgraphProcessor, MaterialGrounding
from src.agents import ResearchAnalyst, ResearchManager, ResearchScientist, RejectedCandidateTracker, MaterialScientist, MultiAnalyst
from src.pipelines import run_material_discovery_pipeline

# ChromaDB and Graph imports
from chromadb import PersistentClient
from autogen.agentchat.contrib.vectordb.chromadb import ChromaVectorDB
import networkx as nx
from sentence_transformers import SentenceTransformer
from GraphReasoning import load_embeddings

# Load configuration
config = load_config()
print("✓ Configuration loaded")


✓ Configuration loaded


## Initialize LLM and Embeddings


In [2]:
# Initialize LLM wrapper
llm_config = {
    "api_key": config["llm"]["api_key"],
    "base_url": config["llm"]["base_url"],
    "model": config["llm"]["model_name"],
    "max_tokens": config["llm"]["max_tokens"]
}
llm_instance = llm(llm_config)
generate = llm_instance.generate_cli
print("✓ LLM wrapper initialized")

# Initialize embedding model
embedding_tokenizer = ''
embedding_model = SentenceTransformer(config["embeddings"]["model_name"], trust_remote_code=True)
embedding_function = TransformerEmbeddingFunction(
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model
)
print("✓ Embedding model initialized")


✓ LLM wrapper initialized


<All keys matched successfully>


✓ Embedding model initialized


## Load Knowledge Graphs and Embeddings


In [3]:
# Load Patent Knowledge Graph (Second KG)
graphs_cfg = config["data"]["graphs"]
kg_dir = graphs_cfg["kg_dir"]

patent_cfg = graphs_cfg["patents"]
patent_kg_path = os.path.join(kg_dir, patent_cfg["graph_file"])
G_patents = nx.read_graphml(patent_kg_path)
relation_patents = nx.get_edge_attributes(G_patents, "title")
nx.set_edge_attributes(G_patents, relation_patents, "relation")
nx.set_node_attributes(G_patents, nx.pagerank(G_patents), "pr")
print(f"Patent KG loaded: {G_patents}")

# Load Patent KG node embeddings
patent_embeddings_path = os.path.join(kg_dir, patent_cfg["embedding_file"])
node_embeddings_patents = load_embeddings(patent_embeddings_path)
print(f"✓ Loaded Patent KG embeddings: {len(node_embeddings_patents)} nodes")

# Load Material Properties Knowledge Graph (Primary KG)
mp_cfg = graphs_cfg["material_properties"]
mp_graph_path = os.path.join(kg_dir, mp_cfg["graph_file"])
G_materialproperties = nx.read_graphml(mp_graph_path)
relation = nx.get_edge_attributes(G_materialproperties, "title")
nx.set_edge_attributes(G_materialproperties, relation, "relation")
nx.set_node_attributes(G_materialproperties, nx.pagerank(G_materialproperties), "pr")
print(f"Material Properties KG loaded: {G_materialproperties}")

# Load Material Properties KG node embeddings
mp_embeddings_path = os.path.join(kg_dir, mp_cfg["embedding_file"])
node_embeddings_materialproperties = load_embeddings(mp_embeddings_path)
print(f"✓ Loaded Material Properties KG embeddings: {len(node_embeddings_materialproperties)} nodes")

print("✓ Both knowledge graphs and embeddings loaded")
print("✓ Dual-KG mode enabled: Will query both Material Properties and Patent knowledge graphs")


Patent KG loaded: DiGraph with 199008 nodes and 578361 edges
✓ Loaded Patent KG embeddings: 199008 nodes
Material Properties KG loaded: DiGraph with 317800 nodes and 774009 edges
✓ Loaded Material Properties KG embeddings: 317800 nodes
✓ Both knowledge graphs and embeddings loaded
✓ Dual-KG mode enabled: Will query both Material Properties and Patent knowledge graphs


## Load ChromaDB Corpus


In [4]:
# Connect to ChromaDB - Load both Patents and MaterialDB ChromaDBs
chroma_cfg = config["data"]["chromadb"]
base_path = chroma_cfg.get("base_path", "")

# Patents ChromaDB
patents_cfg = chroma_cfg["patents"]
if base_path:
    patents_db_path = os.path.join(base_path, patents_cfg["database_path"])
else:
    patents_db_path = patents_cfg["database_path"]

client_patents = PersistentClient(path=patents_db_path)
patents_collections = client_patents.list_collections()
patents_collection_name = patents_cfg["collection_name"] or patents_collections[0].name
patents_collection = client_patents.get_collection(patents_collection_name, embedding_function=embedding_function)

# MaterialDB ChromaDB
materialdb_cfg = chroma_cfg["materialdb"]
if base_path:
    materialdb_db_path = os.path.join(base_path, materialdb_cfg["database_path"])
else:
    materialdb_db_path = materialdb_cfg["database_path"]

client_materialdb = PersistentClient(path=materialdb_db_path)
materialdb_collections = client_materialdb.list_collections()
materialdb_collection_name = materialdb_cfg["collection_name"] or materialdb_collections[0].name
materialdb_collection = client_materialdb.get_collection(materialdb_collection_name, embedding_function=embedding_function)

print(f"✓ Both ChromaDBs loaded and ready to use")


✓ Both ChromaDBs loaded and ready to use


## Initialize Agents


In [ ]:
# Initialize agents
pipeline_cfg = config["pipelines"]["material_discovery"]

# Create chat logger for this run
import uuid
from datetime import datetime

run_id = str(uuid.uuid4())
chat_logger = ChatLogger(
    run_id=run_id,
    pipeline="material_discovery"
)
print(f"✓ Chat logger initialized (run_id: {run_id})")

# Create two ResearchAnalyst instances - one for each ChromaDB
analyst_patents = ResearchAnalyst(
    collection=patents_collection,
    embedding_function=embedding_function,
    n_results=5,
    chat_logger=chat_logger
)

analyst_materialdb = ResearchAnalyst(
    collection=materialdb_collection,
    embedding_function=embedding_function,
    n_results=5,
    chat_logger=chat_logger
)

# Create MultiAnalyst wrapper to query both analysts with source tagging
analyst = MultiAnalyst({
    "patents": analyst_patents,
    "materialdb": analyst_materialdb
})

print("✓ Created ResearchAnalyst instances for Patents and MaterialDB ChromaDBs")
print("✓ MultiAnalyst wrapper created to query both sources with source tagging")

manager = ResearchManager(
    name="research_manager",
    system_message=None,
    generate_fn=generate,
    chat_logger=chat_logger
)

# Initialize ResearchScientist with BOTH knowledge graphs using "separate" strategy
scientist = ResearchScientist(
    knowledge_graph=G_materialproperties,
    node_embeddings=node_embeddings_materialproperties,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    algorithm="shortest",
    chat_logger=chat_logger,
    generate_fn=generate,  # Enable LLM-based material classification instead of keyword-based filtering
    # Second knowledge graph parameters
    knowledge_graph_2=G_patents,
    node_embeddings_2=node_embeddings_patents,
    kg_names=["material_properties", "patents"],
    kg_descriptions=[
        "Material properties and characteristics knowledge graph containing relationships between materials, properties, and applications",
        "Patent knowledge graph containing information about patents, materials, and related technologies"
    ],
    multi_kg_strategy="separate"  # Keep results from each KG separate with descriptions
)

tracker = RejectedCandidateTracker()

print("✓ All agents initialized")
print("✓ ResearchScientist configured with dual-KG mode (Material Properties KG + Patent KG, separate strategy)")

# Initialize new components for re-engineered Step 1
from src.utils import MaterialDatabase, PropertyMapper, SubgraphProcessor, MaterialGrounding
from src.agents import MaterialScientist

# Initialize property mapper for embedding-based property name mapping
property_mapper = PropertyMapper(
    embedding_model=embedding_model,
    embedding_tokenizer=embedding_tokenizer
)
print("✓ Property mapper initialized")

# Load material database
material_db_cfg = config.get("data", {}).get("material_database", {})
material_db_path = material_db_cfg.get("path", "./data/internal_material_database.json")
material_db = MaterialDatabase.load_from_json(material_db_path, property_mapper=property_mapper)
print(f"✓ Material database loaded: {len(material_db)} materials")

# Initialize subgraph processor (uses config utils.subgraph_processor)
subgraph_processor = SubgraphProcessor(embedding_model=embedding_model)
print("✓ Subgraph processor initialized")

# Initialize material grounding
material_grounding = MaterialGrounding(
    knowledge_graph=G_materialproperties,
    node_embeddings=node_embeddings_materialproperties,
    embedding_model=embedding_model,
    embedding_tokenizer=embedding_tokenizer
)
print("✓ Material grounding initialized")

# Initialize substitution reasoner
material_scientist = MaterialScientist(
    material_db=material_db,
    knowledge_graph=G_materialproperties,
    analyst=analyst,
    manager=manager,
    embedding_model=embedding_model,
    embedding_tokenizer=embedding_tokenizer
)
print("✓ Material scientist initialized")


✓ Chat logger initialized (run_id: 4580007d-b698-4bd9-bdbd-809379d7df06)
✓ Created ResearchAnalyst instances for Patents and MaterialDB ChromaDBs
✓ MultiAnalyst wrapper created to query both sources with source tagging
✓ ResearchScientist initialized with 2 knowledge graphs: ['material_properties', 'patents']
  Strategy: separate
  KG Descriptions: ['Material properties and characteristics knowledge graph containing relationships between materials, properties, and applications', 'Patent knowledge graph containing information about patents, materials, and related technologies']
✓ All agents initialized
✓ ResearchScientist configured with dual-KG mode (Material Properties KG + Patent KG, separate strategy)
✓ Property mapper initialized
✓ Material database loaded: 20 materials
✓ Subgraph processor initialized
✓ Material grounding initialized
✓ Material scientist initialized


## Load Properties W from System 1 (or Manual Input)


In [6]:
# Option 1: Load from System 1 output file
# Find the most recent System 1 file
import glob

# Find the most recent System 1 file
system1_files = glob.glob("./pipeline_logs/system1_*.json")
if system1_files:
    # Sort by modification time, get most recent
    system1_files.sort(key=os.path.getmtime, reverse=True)
    system1_output_path = system1_files[0]
    print(f"Using most recent System 1 file: {system1_output_path}")
else:
    raise FileNotFoundError("No System 1 output files found in ./pipeline_logs/")

with open(system1_output_path, "r", encoding="utf-8") as f:
    system1_data = json.load(f)

# Extract base run_id from System 1 file
system1_run_id = system1_data.get("run_id")
if system1_run_id:
    # Extract base part (YYYYMMDDHH) from System 1 run_id
    # Format: YYYYMMDDHH_N -> extract YYYYMMDDHH
    parts = system1_run_id.split("_")
    if len(parts) >= 2:
        base_run_id = parts[0]  # Just the timestamp part (YYYYMMDDHH)
    else:
        base_run_id = system1_run_id  # Fallback to full run_id if format unexpected
    print(f"✓ Extracted base run_id from System 1: {base_run_id} (full: {system1_run_id})")
else:
    # Fallback: generate new base run_id if not found in System 1 file
    from datetime import datetime
    now = datetime.now()
    base_run_id = now.strftime("%Y%m%d%H")
    print(f"⚠ No run_id found in System 1 file, using generated base: {base_run_id}")

properties_W = system1_data["properties_W"]
subgraph_data = system1_data.get("subgraph")  # Load subgraph from System 1
keywords = system1_data.get("keywords", []) or []

# Extract material_X and application_Y directly from System 1 output
# System 1 stores these explicitly, or extract from keywords if not stored
material_X = system1_data.get("material_X")
application_Y = system1_data.get("application_Y")

# Fallback: extract from keywords if not explicitly stored
# Convention: keywords = [material_X, application_Y]
if not material_X and len(keywords) >= 1:
    material_X = str(keywords[0]).strip()
if not application_Y and len(keywords) >= 2:
    application_Y = str(keywords[1]).strip()

# Final fallbacks
if not material_X:
    material_X = "PTFE"  # Default fallback
if not application_Y:
    application_Y = "industrial seals applications"  # Default fallback

sentence = system1_data.get("sentence", "") or ""

# Constraints (empty by default unless you set them)
constraints_U = []

print("=" * 70)
print("Input for System 2:")
print("=" * 70)
print(f"Material to replace (X): {material_X}")
print(f"Application (Y): {application_Y}")
print(f"Required Properties: {properties_W.get('required', [])}")
print(f"Constraints: {len(constraints_U)} constraint(s)")

if subgraph_data:
    print(f"Subgraph: Available ({len(subgraph_data.get('matched_node_ids', []))} matched nodes)")
else:
    print("Subgraph: Not available (will use empty subgraph)")

# Update chat_logger with extracted run_id (will be used for counter logic later)
# Note: chat_logger was initialized in cell 10, but we update it here with System 1 base_run_id
if 'chat_logger' in globals() and 'base_run_id' in globals():
    # Find next available counter for System 2 with this base run_id
    output_dir = "./pipeline_logs"
    pattern = os.path.join(output_dir, f"system2_{base_run_id}_*.json")
    existing_system2_files = glob.glob(pattern)
    
    # Extract max counter
    max_counter = -1
    for file in existing_system2_files:
        filename = os.path.basename(file)
        # Format: system2_YYYYMMDDHH_N.json
        parts = filename.replace(".json", "").split("_")
        if len(parts) >= 3:  # system2_YYYYMMDDHH_N
            try:
                counter = int(parts[-1])
                max_counter = max(max_counter, counter)
            except ValueError:
                pass
    
    system2_counter = max_counter + 1
    system2_run_id = f"{base_run_id}_{system2_counter}"
    chat_logger.run_id = system2_run_id
    print(f"✓ Updated chat_logger run_id: {system2_run_id}")

Using most recent System 1 file: ./pipeline_logs/system1_2026021912_0.json
✓ Extracted base run_id from System 1: 2026021912 (full: 2026021912_0)
Input for System 2:
Material to replace (X): PTFE
Application (Y): industrial seals
Required Properties: ['**Key material properties, performance targets, constraints, and critical characteristics for a PTFE‑E substitute**', 'Continuous operating temperature ≥\u202f250\u202f°C', 'Transient peak tolerance ≥\u202f280–290\u202f°C', 'Thermal degradation onset >\u202f350\u202f°C (mass loss <\u202f1\u202f% up to 350\u202f°C)', 'Glass‑transition far below service temperatures (≈\u202f−80\u202f°C for PTFE)', 'Negligible creep at 250\u202f°C', 'Resistance to concentrated acids (H₂SO₄, HNO₃, HF)', 'Hydrocarbon resistance (alkanes, aromatics)', 'Solvent resistance (acetone, alcohols, ketones, chlorinated solvents)', 'Oxidizer tolerance (H₂O₂, O₃, peracids)', 'High‑temperature stability up to 300\u202f°C', 'UV and ionizing radiation resistance', 'Low fri

## Run System 2: Material Discovery


In [7]:
# Run the material discovery pipeline with re-engineered Step 1
result = run_material_discovery_pipeline(
    material_X=material_X,  # New required parameter
    application_Y=application_Y,
    properties_W=properties_W,
    constraints_U=constraints_U,
    analyst=analyst,
    manager=manager,
    scientist=scientist,
    tracker=tracker,
    max_iterations=pipeline_cfg["max_iterations"],
    temperature=config["llm"]["temperature"],
    chat_logger=chat_logger,
    # Re-engineered Step 1 components
    subgraph_data=subgraph_data,
    material_db=material_db,
    subgraph_processor=subgraph_processor,
    material_grounding=material_grounding,
    material_scientist=material_scientist,
    knowledge_graph=G_materialproperties
)


Material Discovery Pipeline: Starting Process

Pipeline Configuration:
  [OK] Agents verified
  [OK] Material to replace (X): PTFE
  [OK] Application (Y): industrial seals
  [OK] Required Properties: ['**Key material properties, performance targets, constraints, and critical characteristics for a PTFE‑E substitute**', 'Continuous operating temperature ≥\u202f250\u202f°C', 'Transient peak tolerance ≥\u202f280–290\u202f°C', 'Thermal degradation onset >\u202f350\u202f°C (mass loss <\u202f1\u202f% up to 350\u202f°C)', 'Glass‑transition far below service temperatures (≈\u202f−80\u202f°C for PTFE)', 'Negligible creep at 250\u202f°C', 'Resistance to concentrated acids (H₂SO₄, HNO₃, HF)', 'Hydrocarbon resistance (alkanes, aromatics)', 'Solvent resistance (acetone, alcohols, ketones, chlorinated solvents)', 'Oxidizer tolerance (H₂O₂, O₃, peracids)', 'High‑temperature stability up to 300\u202f°C', 'UV and ionizing radiation resistance', 'Low friction coefficient μ ≤\u202f0.13 under expected load

## Display Results and Save


In [8]:
# Display results
print("\n" + "=" * 70)
print("Material Discovery Results Summary")
print("=" * 70)

if result.get("success"):
    print("✓ SUCCESS: Viable candidate material found!")
    candidate = result.get("candidate", {})
    print(f"\nCandidate Material: {candidate.get('material_name', 'Unknown')}")
    print(f"Material Class: {candidate.get('material_class', 'Unknown')}")
    print(f"Iterations Required: {result.get('iterations', 0)}")
    print(f"\nJustification:")
    print(candidate.get('justification', '')[:500])
else:
    print("✗ No viable candidate found")
    print(f"Iterations attempted: {result.get('iterations', 0)}")
    print(f"Rejected candidates: {len(result.get('rejected_candidates', []))}")
    if result.get('final_constraints'):
        print(f"\nFinal constraints that prevented success:")
        for constraint in result.get('final_constraints', [])[:5]:
            print(f"  - {constraint}")

# Display property mappings
property_mapping = result.get('property_mapping', {})
if property_mapping:
    prop_mappings = property_mapping.get('property_mappings', {})
    if prop_mappings:
        print(f"\n\nProperty Mappings:")
        print("-" * 70)
        print("The following properties were mapped from database property names to target properties:")
        print(f"\n{'Target Property':<50} {'Database Properties':<60}")
        print(f"{'-'*50} {'-'*60}")
        for target_prop, db_props in prop_mappings.items():
            db_prop_list = list(db_props.keys())
            # Truncate target property if too long
            target_display = target_prop[:47] + "..." if len(target_prop) > 50 else target_prop
            # Show first few DB properties
            if db_prop_list:
                db_display = ", ".join(db_prop_list[:4])
                if len(db_prop_list) > 4:
                    db_display += f" ... (+{len(db_prop_list) - 4} more)"
                print(f"{target_display:<50} {db_display:<60}")
            else:
                print(f"{target_display:<50} {'(no mappings found)':<60}")

# Display iteration history
if result.get('iteration_history'):
    print(f"\n\nIteration History:")
    print("-" * 70)
    for hist in result.get('iteration_history', []):
        iter_num = hist.get('iteration', 0)
        candidate_name = hist.get('candidate', 'Unknown')
        feasible = hist.get('feasible', False)
        status = "✓ FEASIBLE" if feasible else "✗ REJECTED"
        print(f"Iteration {iter_num}: {candidate_name} - {status}")
        if not feasible and hist.get('constraints_violated'):
            print(f"  Constraints violated: {', '.join(hist.get('constraints_violated', [])[:3])}")

# Save results
# Use run_id from chat_logger (which should have been set from System 1 base_run_id + counter)
if chat_logger and hasattr(chat_logger, 'run_id') and chat_logger.run_id:
    result_run_id = chat_logger.run_id
elif 'base_run_id' in globals():
    # Find next available counter for System 2 with this base run_id
    output_dir = "./pipeline_logs"
    pattern = os.path.join(output_dir, f"system2_{base_run_id}_*.json")
    existing_system2_files = glob.glob(pattern)
    
    # Extract max counter
    max_counter = -1
    for file in existing_system2_files:
        filename = os.path.basename(file)
        # Format: system2_YYYYMMDDHH_N.json
        parts = filename.replace(".json", "").split("_")
        if len(parts) >= 3:  # system2_YYYYMMDDHH_N
            try:
                counter = int(parts[-1])
                max_counter = max(max_counter, counter)
            except ValueError:
                pass
    
    system2_counter = max_counter + 1
    result_run_id = f"{base_run_id}_{system2_counter}"
else:
    # Fallback: generate new run_id
    from datetime import datetime
    now = datetime.now()
    base_id = now.strftime("%Y%m%d%H")
    result_run_id = f"{base_id}_0"

timestamp = datetime.utcnow().isoformat() + "Z"

output_dir = "./pipeline_logs"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"system2_{result_run_id}.json")

discovery_payload = {
    "run_id": result_run_id,
    "timestamp": timestamp,
    "application": application_Y,
    "properties": properties_W,
    "constraints": constraints_U,
    "success": result.get("success", False),
    "candidate": result.get("candidate"),
    "iterations": result.get("iterations", 0),
    "rejected_candidates": result.get("rejected_candidates", []),
    "final_constraints": result.get("final_constraints", []),
    "evidence_summary": result.get("evidence_summary", {}),
    "iteration_history": result.get("iteration_history", []),
    "property_mapping": result.get("property_mapping", {}),
    "chat_log_path": result.get("chat_log_path")
}

with open(output_path, "w") as f:
    json.dump(discovery_payload, f, indent=2)

print(f"\n✓ Saved material discovery results to {output_path}")

# Save chat log if available
if chat_logger:
    try:
        chat_log_path = chat_logger.save()
        if chat_log_path:
            print(f"✓ Saved chat log to {chat_log_path}")
            print(f"  Use this file in chat_visualization.ipynb to visualize System 2 chat interactions")
    except Exception as e:
        print(f"Warning: Failed to save chat log: {e}")



Material Discovery Results Summary
✓ SUCCESS: Viable candidate material found!

Candidate Material: BASF PBI‑ELAST 2000
Material Class: Unknown
Iterations Required: 1

Justification:
Material Name: BASF PBI‑ELAST 2000  
Justification:  
BASF PBI‑ELAST 2000 is a high‑temperature polybenzimidazole (PBI) elastomer that meets every requirement for an industrial seal PTFE‑E substitute.  Its aromatic backbone gives a thermal degradation onset above 350 °C with <1 % mass loss up to 350 °C, satisfying the continuous operating temperature of ≥250 °C and transient peak tolerance of 280–290 °C.  The elastomeric grade has a glass‑transition around –80 °C (due to flexible side chains) so


Iteration History:
----------------------------------------------------------------------
Iteration 1: BASF PBI‑ELAST 2000 - ✓ FEASIBLE

✓ Saved material discovery results to ./pipeline_logs/system2_2026021912_0.json
✓ Saved chat log to ./pipeline_logs/chats/system2_chat_log_material_discovery_2026021912_0.json
